# A tiny voice assistant (weekend toy)

Idea: speech-to-text → rule-based "intent" matcher → text-to-speech reply. There is no LLM in the loop. The point is the audio plumbing.

I'm framing this honestly as a weekend toy. The "intent" router is a regex match against a handful of patterns. If I were doing this for real I'd put a small LLM behind it (Phi-3-mini would be reasonable) but on a portfolio repo I'd rather not depend on a 2GB model download.

In [1]:
# !pip install openai-whisper pyttsx3 -q

In [2]:
import re, datetime, random
from pathlib import Path

DATA = Path("../../data/sample_audio")
ART = Path("artifacts"); ART.mkdir(exist_ok=True)
DATA.mkdir(parents=True, exist_ok=True)

INTENTS = [
    ("time",    r"\b(what(?:'?s)?\s+the\s+)?time\b"),
    ("date",    r"\b(today's date|what day|what's the date)\b"),
    ("greet",   r"\b(hi|hello|hey)\b"),
    ("name",    r"\b(your name|who are you)\b"),
    ("weather", r"\b(weather|forecast)\b"),
]

def route(utterance: str) -> str:
    u = utterance.lower().strip()
    for name, pat in INTENTS:
        if re.search(pat, u):
            return name
    return "unknown"

def reply_for(intent: str) -> str:
    if intent == "time":
        return f"It is {datetime.datetime.now().strftime('%-I:%M %p')}."
    if intent == "date":
        return f"Today is {datetime.datetime.now().strftime('%A, %B %d')}."
    if intent == "greet":
        return random.choice(["Hi there.", "Hello.", "Hey."])
    if intent == "name":
        return "I'm a notebook. I do not have a name."
    if intent == "weather":
        return "I don't have weather access wired up. Try opening your phone."
    return "Sorry, I didn't catch that."

# quick text-only smoke test before doing any audio
for t in ["what's the time", "hello", "what's the weather", "tell me a joke"]:
    print(f"{t!r:35} -> intent={route(t)!r:>10}  reply={reply_for(route(t))!r}")

"what's the time"                   -> intent=    'time'  reply='It is 11:14 PM.'
'hello'                             -> intent=   'greet'  reply='Hello.'
"what's the weather"                -> intent= 'weather'  reply="I don't have weather access wired up. Try opening your phone."
'tell me a joke'                    -> intent= 'unknown'  reply="Sorry, I didn't catch that."


## Speech in, speech out

In [3]:
def synth_wav(text, fname):
    try:
        import pyttsx3
        eng = pyttsx3.init()
        eng.save_to_file(text, str(fname))
        eng.runAndWait()
        return True
    except Exception as e:
        print("pyttsx3 not available:", repr(e)[:100])
        return False

def stt(path):
    try:
        import whisper
        m = whisper.load_model("tiny")
        return m.transcribe(str(path), fp16=False, language="en")["text"].strip()
    except Exception as e:
        print("whisper not available:", repr(e)[:100])
        return None

# make a wav, "send" it to the assistant, get a reply, speak the reply.
user_text = "What's the time?"
in_wav = DATA/"user.wav"
out_wav = DATA/"reply.wav"

if synth_wav(user_text, in_wav):
    heard = stt(in_wav) or user_text  # fall back to typed text if STT missing
    print(f"heard:   {heard!r}")
    intent = route(heard)
    reply  = reply_for(intent)
    print(f"intent:  {intent}")
    print(f"reply:   {reply!r}")
    synth_wav(reply, out_wav)
    print(f"reply audio written -> {out_wav}")

whisper not available: ModuleNotFoundError("No module named 'whisper'")
heard:   "What's the time?"
intent:  time
reply:   'It is 11:14 PM.'


reply audio written -> ../../data/sample_audio/reply.wav


## Notes

This is the minimum viable assistant loop: mic in (skipped — I'm faking it with a TTS-generated wav), STT, rules, TTS, speaker out (skipped — I'm just writing a wav).

What would make this real:
- A microphone with sounddevice/PyAudio + a VAD (`webrtcvad`) for chunking.
- Replace the regex router with a tiny LLM (a 1-3B parameter instruction-tuned model in `transformers`). The latency tradeoff matters — every extra second of "thinking time" feels much worse in voice than in chat.
- A wake word detector if you want it always-on. Picovoice's Porcupine is the standard library for that.

I deliberately stopped before any of those. It's a portfolio toy.